In [1]:
import oceanbench

oceanbench.__version__

'0.2.0'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# Open GLONET forecast sample with xarray
from datetime import datetime
import xarray
from oceanbench.core.interpolate import interpolate_1_degree

challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [
        "https://s3.waw3-1.cloudferro.com/oceanbench-bucket/public/ml-forecast-outputs/glonet/20240103.zarr",
        "https://s3.waw3-1.cloudferro.com/oceanbench-bucket/public/ml-forecast-outputs/glonet/20240110.zarr",
    ],
    engine="zarr",
    preprocess=lambda dataset: dataset.rename({"time": "lead_day_index"}).assign({"lead_day_index": range(10)}),
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=True,
).assign(
    {
        "first_day_datetime": [
            datetime.fromisoformat("2024-01-03"),
            datetime.fromisoformat("2024-01-10"),
        ]
    }
)
challenger_dataset = interpolate_1_degree(challenger_dataset)
challenger_dataset


<xarray.Dataset> Size: 823MB
Dimensions:                          (first_day_datetime: 2,
                                      lead_day_index: 10, depth: 21,
                                      latitude: 168, longitude: 360)
Coordinates:
  * depth                            (depth) float32 84B 0.494 ... 5.275e+03
  * lead_day_index                   (lead_day_index) int64 80B 0 1 2 ... 7 8 9
  * first_day_datetime               (first_day_datetime) datetime64[us] 16B ...
  * latitude                         (latitude) float64 1kB -77.5 -76.5 ... 89.5
  * longitude                        (longitude) float64 3kB -179.5 ... 179.5
Data variables:
    sea_water_salinity               (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 2, 1, 168, 360), meta=np.ndarray>
    sea_water_potential_temperature  (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 2, 1, 168, 360), meta=np.ndarray>
    eastward_sea_water_velocity      (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 2, 1, 168, 360), meta=np.ndarray>
    northward_sea_water_velocity     (first_day_datetime, lead_day_index, depth, latitude, longitude) float64 203MB dask.array<chunksize=(1, 2, 1, 168, 360), meta=np.ndarray>
    sea_surface_height_above_geoid   (first_day_datetime, lead_day_index, latitude, longitude) float64 10MB dask.array<chunksize=(1, 2, 168, 360), meta=np.ndarray>
Attributes:
    Conventions:              CF-1.8
    area:                     Global
    challenger:               glonet
    contact:                  glonet@mercator-ocean.eu
    forecast_reference_time:  2024-01-02
    institution:              Mercator Ocean International
    references:               www.edito.eu
    source:                   MOI GLONET
    title:                    Daily mean fields from GLONET 1/4 degree resolu...

### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.073326,0.074905,0.075668,0.077249,0.080264,0.083001,0.084822,0.087562,0.089371,0.092031
Temperature (°C) [sea_water_potential_temperature]{surface},0.652544,0.653534,0.695322,0.698382,0.749915,0.766597,0.817991,0.834906,0.871502,0.885801
Salinity (PSU) [sea_water_salinity]{surface},0.669255,0.672911,0.664816,0.667342,0.653718,0.656344,0.644467,0.646526,0.637756,0.642323
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.117016,0.119472,0.123207,0.123589,0.127067,0.128123,0.132276,0.133803,0.137357,0.137242
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119212,0.121291,0.124858,0.126396,0.129628,0.130662,0.134467,0.135599,0.138290,0.139478
Temperature (°C) [sea_water_potential_temperature]{50m},0.946886,0.952772,0.972768,0.982616,1.016913,1.032786,1.076040,1.096520,1.137864,1.159819
Salinity (PSU) [sea_water_salinity]{50m},0.344726,0.345823,0.349519,0.351977,0.352621,0.355776,0.357247,0.360620,0.362634,0.365919
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.108906,0.109511,0.109378,0.110108,0.111544,0.113383,0.116291,0.118711,0.121105,0.121893
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.110756,0.111027,0.110831,0.111942,0.113466,0.115250,0.117607,0.119791,0.122214,0.123791
Temperature (°C) [sea_water_potential_temperature]{100m},1.041536,1.050870,1.059343,1.068462,1.088487,1.098582,1.125019,1.137147,1.165924,1.179266


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},45.931473,46.225616,47.074535,47.639011,49.046417,49.925003,50.918697,52.032276,52.859642,53.417297


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.056713,0.058406,0.062245,0.065422,0.067399,0.072046,0.074301,0.080590,0.080758,0.088228
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.053942,0.058423,0.061723,0.064942,0.066368,0.069746,0.074616,0.078057,0.079956,0.085047


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Temperature (°C) [sea_water_potential_temperature]{surface},0.740377,0.829552,0.823272,0.796854,0.797869,0.823776,0.888996,0.912615,0.967003,0.891948
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.624492,0.620448,0.684111,0.632325,0.641032,0.750998,0.820515,0.828752,0.812411,0.888377
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.911723,1.082018,1.041187,1.007146,1.113241,1.004200,1.039810,1.263613,1.085817,1.292335
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.933362,0.948271,0.864101,0.889127,0.839232,0.940400,0.996264,0.965732,0.919232,0.975473
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.535094,0.549522,0.520502,0.508881,0.498371,0.574513,0.603877,0.561134,0.509138,0.596548
Salinity (PSU) [sea_water_salinity]{0-5m},0.220809,0.195206,0.222592,0.280205,0.227698,0.226738,0.215709,0.239103,0.232819,0.257623
Salinity (PSU) [sea_water_salinity]{5-100m},0.190362,0.175902,0.197807,0.175506,0.197721,0.215337,0.222479,0.199656,0.199856,0.214708
Salinity (PSU) [sea_water_salinity]{100-300m},0.124787,0.131037,0.118239,0.122506,0.114879,0.149576,0.143243,0.121583,0.125763,0.144141
Salinity (PSU) [sea_water_salinity]{300-600m},0.073906,0.081524,0.075358,0.072259,0.080508,0.083347,0.087264,0.089482,0.076631,0.084903
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.107485,0.108930,0.108813,0.109382,0.109516,0.110718,0.109506,0.109085,0.109259,0.109827


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},9.609055,17.764639,25.076084,31.854841,38.369987,44.708733,50.91555,57.069878


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.027169,0.032395,0.035664,0.039895,0.046811,0.052053,0.056360,0.061349,0.063894,0.066508
Temperature (°C) [sea_water_potential_temperature]{surface},0.411497,0.421295,0.506956,0.520767,0.602565,0.623731,0.695742,0.723394,0.771373,0.787303
Salinity (PSU) [sea_water_salinity]{surface},0.158379,0.159919,0.209747,0.213745,0.259533,0.263154,0.300415,0.300577,0.334210,0.331464
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.048498,0.066422,0.080607,0.089914,0.099537,0.107251,0.118504,0.124982,0.132087,0.135192
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.048237,0.066672,0.079779,0.091028,0.102919,0.110994,0.120553,0.127478,0.133223,0.136707
Temperature (°C) [sea_water_potential_temperature]{50m},0.720263,0.720665,0.764706,0.781302,0.848141,0.872878,0.942729,0.972223,1.032644,1.054387
Salinity (PSU) [sea_water_salinity]{50m},0.108986,0.109339,0.139972,0.141450,0.164220,0.165739,0.184140,0.186906,0.202272,0.202858
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.041427,0.051060,0.060149,0.067293,0.075756,0.083140,0.092122,0.099668,0.106147,0.110404
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.042288,0.051866,0.061203,0.069161,0.077582,0.085534,0.093860,0.101071,0.106719,0.111571
Temperature (°C) [sea_water_potential_temperature]{100m},0.538064,0.549862,0.616528,0.646908,0.732452,0.769726,0.849117,0.887400,0.941958,0.971052


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},41.719681,41.904816,43.827648,43.963654,45.914711,46.860706,48.42849,49.35321,50.394356,50.742905


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Lead day 10
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.034642,0.039341,0.045589,0.050459,0.055842,0.062475,0.067280,0.075684,0.076725,0.085118
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.035578,0.043491,0.048340,0.051970,0.056136,0.062426,0.070003,0.074757,0.075837,0.081456


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Lagrangian trajectory deviation (km) []{surface},4.407306,9.138875,13.983825,18.947304,24.068195,29.537165,35.28125,41.133686
